# First MVP RL

In [6]:
import numpy as np
import random
from collections import defaultdict

# Class Environment

In [7]:
class SmartChargingEnv:
    """
    Discrete event simulation for the EV charging problem.
    """
    def __init__(self, capacity=50, max_power=22):
        self.capacity = capacity
        self.max_power = max_power
        
        # Actions: 0 (Zero), 1 (Low: 7kW), 2 (Medium: 11kW), 3 (High: 22kW)
        self.action_space = [0, 7, 11, 22] 
        self.max_steps = 8 # 2:00 PM to 4:00 PM in 15-min intervals
        
        # Environment state
        self.current_step = 0
        self.battery_level = 0.0
        
        # Time coefficients for cost (simulating variable electricity pricing)
        # Randomly initialized for scaffolding; replace with real data if available
        self.alpha_t = [random.uniform(0.1, 0.5) for _ in range(self.max_steps)]

    def reset(self):
        self.current_step = 0
        # Assume battery starts empty or at a random low value when arriving at 2 p.m.
        self.battery_level = 0.0 
        return self._get_state()

    def _get_state(self):
        # Discretize battery level to integer for basic Q-Learning compatibility
        return (self.current_step, int(self.battery_level))

    def step(self, action_idx):
        power_kw = self.action_space[action_idx]
        
        # 1. Calculate new battery level (Power * 0.25 hours)
        energy_added = power_kw * 0.25
        self.battery_level = min(self.capacity, self.battery_level + energy_added)
        
        # 2. Calculate immediate charging cost (Exponential cost function)
        alpha = self.alpha_t[self.current_step]
        # Using a scaled exponent so math.exp doesn't blow up with 22kW
        charging_cost = alpha * np.exp(power_kw / 10.0) 
        reward = -charging_cost
        
        self.current_step += 1
        
        # 3. Check termination and apply demand penalty
        done = False
        if self.current_step >= self.max_steps:
            done = True
            # Stochastic demand generation: Normal dist (mu=30, sigma=5)
            demand = np.random.normal(30, 5)
            
            if self.battery_level < demand:
                # High penalty for running out of energy
                reward -= 1000 
                
        return self._get_state(), reward, done

## Class Agent

In [8]:
class QLearningAgent:
    def __init__(self, action_space, lr=0.1, gamma=0.99, epsilon=0.1):
        # Using a dictionary for the Q-table since states are tuples: (step, battery)
        self.q_table = defaultdict(lambda: np.zeros(len(action_space)))
        self.action_space = action_space
        self.lr = lr
        self.gamma = gamma
        self.epsilon = epsilon

    def choose_action(self, state):
        if random.uniform(0, 1) < self.epsilon:
            return random.randint(0, len(self.action_space) - 1)
        return np.argmax(self.q_table[state])

    def learn(self, s, a, r, s_next, done):
        if done:
            td_target = r
        else:
            best_next_action = np.argmax(self.q_table[s_next])
            td_target = r + self.gamma * self.q_table[s_next][best_next_action]
            
        td_error = td_target - self.q_table[s][a]
        self.q_table[s][a] += self.lr * td_error

## --- Execution Scaffolding ---

In [9]:
env = SmartChargingEnv()
agent = QLearningAgent(action_space=env.action_space)

episodes = 5000 # Increased episodes because state space is larger

for i in range(episodes):
    state = env.reset()
    total_reward = 0
    done = False
    
    while not done:
        action_idx = agent.choose_action(state)
        next_state, reward, done = env.step(action_idx)
        
        agent.learn(state, action_idx, reward, next_state, done)
        
        state = next_state
        total_reward += reward

print(f"Training finished. Q-Table has learned {len(agent.q_table)} state-action mappings.")

Training finished. Q-Table has learned 143 state-action mappings.


In [10]:
def evaluate_smart_charging(env, agent, test_episodes=1000):
    print(f"\n--- Starting Evaluation over {test_episodes} simulated days ---")
    
    # 1. Turn off exploration (Pure Exploitation)
    agent.epsilon = 0.0 
    
    failures = 0
    total_charging_cost = 0.0
    
    for _ in range(test_episodes):
        state = env.reset()
        done = False
        episode_cost = 0.0
        
        while not done:
            # 2. Always choose the optimal known action
            action_idx = np.argmax(agent.q_table[state])
            next_state, reward, done = env.step(action_idx)
            
            # Reconstruct the cost from the reward
            # If the reward is massive negative, it hit the penalty, not just a cost
            if reward > -500: 
                episode_cost += abs(reward) # Cost is the absolute value of the negative reward
                
            if done and reward <= -500:
                failures += 1
                
            state = next_state
            
        # Only add to average cost if the agent didn't fail
        if reward > -500: 
            total_charging_cost += episode_cost
            
    # 3. Calculate Final Metrics
    successful_days = test_episodes - failures
    success_rate = (successful_days / test_episodes) * 100
    avg_cost = total_charging_cost / successful_days if successful_days > 0 else 0
    
    print(f"Success Rate (Shift demand met): {success_rate:.2f}%")
    print(f"Failure Rate (Ran out of energy): {(failures/test_episodes)*100:.2f}%")
    if successful_days > 0:
         print(f"Average Charging Cost per day: ${avg_cost:.2f}")
    else:
         print("Agent failed every single day. No average cost calculated.")

# Run the evaluation
evaluate_smart_charging(env, agent, test_episodes=1000)


--- Starting Evaluation over 1000 simulated days ---
Success Rate (Shift demand met): 100.00%
Failure Rate (Ran out of energy): 0.00%
Average Charging Cost per day: $21.01
